# Skill Extraction Model - Exploration Notebook

This notebook demonstrates how to load the SkillSpan dataset, interact with the custom data pipeline (including subword label propagation), instantiate the minimalist transformer, and run a sample forward pass.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import torch
from dataset import get_dataloaders, ID_TO_LABEL
from model import SkillExtractor

## 1. Load Data and Check Label Alignment

In [ ]:
# Load dataloaders
train_loader, val_loader, test_loader, tokenizer = get_dataloaders(batch_size=2, max_length=128)

# Fetch a single batch
batch = next(iter(train_loader))
input_ids = batch["input_ids"][0]
labels = batch["labels"][0]

# Show alignments for the first 30 tokens
tokens = tokenizer.convert_ids_to_tokens(input_ids)
print(f"{'Token':>15} : {'Label'}")
print("-" * 30)
for token, label in zip(tokens[:30], labels[:30]):
    label_str = ID_TO_LABEL.get(label.item(), "IGNORED") if label.item() != -100 else "IGNORED (-100)"
    print(f"{token:>15} : {label_str}")

## 2. Model Initialization and Budget Check

In [ ]:
model = SkillExtractor(
    vocab_size=tokenizer.vocab_size,
    d_model=128,
    num_heads=4,
    num_layers=4,
    d_ff=512,
    num_classes=5,
    max_len=128
)

param_count = model.get_num_parameters()
print(f"Total trainable parameters: {param_count:,}")
print(f"Budget limit (6M): {'✅ PASS' if param_count < 6000000 else '❌ FAIL'}")

## 3. Sample Forward Pass

In [ ]:
model.eval()
with torch.no_grad():
    # Shape (batch_size, seq_len)
    logits = model(batch["input_ids"], batch["attention_mask"])
    predictions = torch.argmax(logits, dim=-1)

print(f"Input shape: {batch['input_ids'].shape}")
print(f"Logits shape: {logits.shape}")
print(f"Predictions shape: {predictions.shape}")

# The output is ready for CrossEntropyLoss!